In [2]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script. cd ~/SPCS
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# BASELINE: no data augmentation at all here - this is the control
# run to compare against the speed/pitch/noise/SpecAugment variants.
# SpecAugment is explicitly turned OFF (rather than left at whatever
# the pretrained config happened to have) so this run is a clean,
# unambiguous "no augmentation" baseline.
model.config.apply_spec_augment = False
model.config.mask_time_prob = 0.0
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.0
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# NOTE ON AUGMENTATION
# BASELINE run - no augmentation of any kind: no waveform
# perturbation (speed/pitch/noise) and no SpecAugment (see
# model.config above, explicitly disabled). Training sees the
# audio exactly as loaded. Use this run's WER/CER as the reference
# point the augmentation variants are meant to beat.
# =========================================================

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    # No waveform-level augmentation here - SpecAugment (applied inside
    # the model, see model.config above) is the sole regularizer.
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_baseline",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_baseline"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_baseline


In [14]:
# =========================================================
# INSTALL DEPENDENCIES
# =========================================================
!pip install -q transformers datasets evaluate jiwer accelerate soundfile huggingface_hub librosa

# =========================================================
# IMPORTS
# =========================================================
import os
import random
import numpy as np
import torch
import pandas as pd
import evaluate
import librosa

from huggingface_hub import login

from dataclasses import dataclass
from typing import Any

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

# =========================================================
# HF LOGIN
# Prompts you to paste your token at runtime (hidden input, like a
# PIN) instead of hardcoding it in the script.
# =========================================================
from getpass import getpass

login(getpass("Enter your Hugging Face token: "))

# =========================================================
# DEVICE
# =========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =========================================================
# PATHS
# =========================================================
train_csv = "SPCS/manifests/train.csv"
valid_csv = "SPCS/manifests/valid.csv"
test_csv = "SPCS/manifests/test.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(train_csv)
valid_df = pd.read_csv(valid_csv)

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

# =========================================================
# DATASET CONVERSION
# =========================================================
train_ds = Dataset.from_pandas(train_df)
valid_ds = Dataset.from_pandas(valid_df)

train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio", Audio(sampling_rate=16000))

# =========================================================
# MODEL + PROCESSOR
# Switch model_name to "openai/whisper-tiny" if small keeps
# overfitting hard even after the regularization changes below.
# =========================================================
model_name = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(model_name, task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = None
model.generation_config.task = "transcribe"

# ---- Regularization additions ----
# BASELINE: no data augmentation at all here - this is the control
# run to compare against the speed/pitch/noise/SpecAugment variants.
# SpecAugment is explicitly turned OFF (rather than left at whatever
# the pretrained config happened to have) so this run is a clean,
# unambiguous "no augmentation" baseline.
model.config.apply_spec_augment = False
model.config.mask_time_prob = 0.0
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.0
model.config.mask_feature_length = 10

# Optional: freeze the encoder if your training set is very small
# (e.g. a few hours of audio). This forces the model to only adapt
# the decoder/language side, which is much less prone to overfitting
# on limited data. Uncomment if train loss still crashes to ~0 fast.
# model.freeze_encoder()

model.to(device)

# =========================================================
# NOTE ON AUGMENTATION
# BASELINE run - no augmentation of any kind: no waveform
# perturbation (speed/pitch/noise) and no SpecAugment (see
# model.config above, explicitly disabled). Training sees the
# audio exactly as loaded. Use this run's WER/CER as the reference
# point the augmentation variants are meant to beat.
# =========================================================

# =========================================================
# PREPROCESSING
# =========================================================
def prepare_dataset_train(batch):
    audio = batch["audio"]

    # No waveform-level augmentation here - SpecAugment (applied inside
    # the model, see model.config above) is the sole regularizer.
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

def prepare_dataset_eval(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

train_ds = train_ds.map(
    prepare_dataset_train,
    remove_columns=train_ds.column_names,
    num_proc=1
)

valid_ds = valid_ds.map(
    prepare_dataset_eval,
    remove_columns=valid_ds.column_names,
    num_proc=1
)

# =========================================================
# DATA COLLATOR
# =========================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

# =========================================================
# WER + CER METRICS
# =========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

# =========================================================
# TRAINING CONFIG (FIXED - anti-overfitting)
# =========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="SPCS/models_result/whisper_baseline",

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,   # effective batch size 8

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,               # smooth ramp-up instead of jumping straight to peak LR
    weight_decay=0.01,              # penalizes large weights, directly fights overfitting

    num_train_epochs=10,            # kept high on purpose - early stopping will cut it short
    max_grad_norm=1.0,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,
    save_total_limit=3,

    predict_with_generate=True,
    generation_max_length=225,

    # Keep the checkpoint with the best validation WER, not the last one.
    # This alone fixes most of the damage from overfitting: even if
    # training runs long, you deploy the epoch that generalized best.
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # NOTE: safe_serialization is handled manually at save time below
    # (this transformers version doesn't accept save_safetensors here).

    report_to="none"
)

# =========================================================
# TRAINER
# =========================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
    # Stops training if validation WER hasn't improved for 3 straight
    # evals - prevents the "train loss -> 0, val loss flat" pattern
    # you saw from running all the way to epoch 10.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

# =========================================================
# TRAIN
# =========================================================
trainer.train()

# =========================================================
# SAVE MODEL (this saves the BEST checkpoint, thanks to
# load_best_model_at_end=True above)
# =========================================================
save_path = "SPCS/models_result/whisper_baseline"

model.tie_weights()  # make sure proj_out is explicitly linked to the embedding weights

# Saving directly via model.save_pretrained (rather than trainer.save_model)
# so we can pass safe_serialization=False - this avoids the "missing keys:
# proj_out.weight" warning caused by Whisper's tied embedding/output layer
# being dropped from safetensors checkpoints on save.
model.save_pretrained(save_path, safe_serialization=False)
processor.save_pretrained(save_path)

# =========================================================
# SAVE TRAINER STATE (training loss / val loss / wer / cer history)
# Writes trainer_state.json into save_path so the full log_history
# is reloadable later, instead of only living in checkpoint-N folders.
# =========================================================
trainer.save_state()

log_history = trainer.state.log_history
log_df = pd.DataFrame(log_history)
log_csv_path = os.path.join(save_path, "training_log_history.csv")
log_df.to_csv(log_csv_path, index=False)

print("\nTraining complete.")
print("Best model (by validation WER) saved to:", save_path)
print("Trainer state saved to:", save_path)
print("Training log history exported to:", log_csv_path)

# =========================================================
# TEST SET EVALUATION (NEW)
# Runs the best saved checkpoint on the held-out test set - data
# never seen during training or used for checkpoint selection.
# This gives the final, unbiased WER/CER for the baseline model.
# =========================================================
test_df = pd.read_csv(test_csv)
print("\nTest samples:", len(test_df))

test_ds = Dataset.from_pandas(test_df)
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds = test_ds.map(prepare_dataset_eval, remove_columns=test_ds.column_names, num_proc=1)

test_results = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")

print("\nTest set results:")
print(test_results)

# ---- Save test results to CSV ----
test_results_df = pd.DataFrame([test_results])
test_results_csv_path = os.path.join(save_path, "test_results.csv")
test_results_df.to_csv(test_results_csv_path, index=False)
print("Test results saved to:", test_results_csv_path)

Enter your Hugging Face token:  ········


Using device: cuda
Training samples: 9908
Validation samples: 1239


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Map:   0%|          | 0/9908 [00:00<?, ? examples/s]

Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

...


Training complete.
Best model (by validation WER) saved to: SPCS/models_result/whisper_baseline
Trainer state saved to: SPCS/models_result/whisper_baseline
Training log history exported to: SPCS/models_result/whisper_baseline/training_log_history.csv

Test samples: 1239


Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled


Training Loss,Validation Loss,Epoch,Wer,Cer
0.000101,0.034664,10,0.077065,0.035904



Test set results:
{'test_loss': 0.03466394543647766, 'test_wer': 0.07706489675516225, 'test_cer': 0.03590401825507184}
Test results saved to: SPCS/models_result/whisper_baseline/test_results.csv


In [16]:
import matplotlib.pyplot as plt
import numpy as np
import os

# ---- Ensure output directory exists ----
output_dir = "SPCS/graph_results/test"
os.makedirs(output_dir, exist_ok=True)

# ---- Data ----
runs = ["Baseline", "Noise\nInjection", "Speed\nPerturbation", "Pitch\nShift", "SpecAugment"]

test_wer = [0.077065, 0.018990, 0.035398, 0.030973, 0.022124]
test_cer = [0.035904, 0.019432, 0.026848, 0.019218, 0.021570]
test_loss = [0.034664, 0.013267, 0.010337, 0.011757, 0.009368]

colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

# ---------------------------------------------------------------
# Figure 4.9: Test WER and CER comparison across all models
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(runs))
width = 0.35

bars1 = ax.bar(x - width/2, np.array(test_wer) * 100, width,
               label="Test WER (%)", color="#4C72B0")
bars2 = ax.bar(x + width/2, np.array(test_cer) * 100, width,
               label="Test CER (%)", color="#DD8452")

ax.set_ylabel("Error rate (%)")
ax.set_title("Test Set WER and CER Comparison Across All Models")
ax.set_xticks(x)
ax.set_xticklabels(runs)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

for bars in (bars1, bars2):
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.2f}", xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "test_wer_cer.png"), dpi=200)
plt.close()

# ---------------------------------------------------------------
# Figure 4.10: Test loss comparison across all models
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(runs, test_loss, color=colors)
ax.set_ylabel("Test Loss")
ax.set_title("Test Loss Comparison Across All Models")
ax.grid(axis="y", linestyle="--", alpha=0.4)

for b in bars:
    h = b.get_height()
    ax.annotate(f"{h:.4f}", xy=(b.get_x() + b.get_width() / 2, h),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "test_loss.png"), dpi=200)
plt.close()

print(f"Saved figures to: {output_dir}")

Saved figures to: SPCS/graph_results/test


In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- Results data ----
data = {
    "Epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Training Loss": [2.156137, 0.485272, 0.137459, 0.014917, 0.004883,
                       0.012100, 0.008397, 0.020184, 0.000111, 0.000098],
    "Validation Loss": [0.243621, 0.076819, 0.044315, 0.036883, 0.032657,
                         0.030163, 0.030880, 0.030177, 0.030742, 0.030304],
    "Wer": [0.287868, 0.090993, 0.055882, 0.041360, 0.031434,
            0.029596, 0.027390, 0.027757, 0.028309, 0.028493],
    "Cer": [0.153373, 0.049223, 0.029702, 0.018892, 0.014729,
            0.013679, 0.012175, 0.012839, 0.012524, 0.013049],
}

df = pd.DataFrame(data)

# Round all numeric columns to 3 decimal places
df = df.round(4)

# ---- Output directory ----
output_dir = "SPCS/graph_results/baseline_whisper"
os.makedirs(output_dir, exist_ok=True)

# Save table as CSV
csv_path = os.path.join(output_dir, "baseline_whisper_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved table to: {csv_path}")

# ---- Save a table image ----
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.title("Baseline Whisper - Training Results", pad=15)
plt.tight_layout()

table_img_path = os.path.join(output_dir, "baseline_whisper_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved table image to: {table_img_path}")

# ---- Save a metrics plot (loss/WER/CER over epochs) ----
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["Epoch"], df["Training Loss"], marker="o", label="Training Loss")
ax1.plot(df["Epoch"], df["Validation Loss"], marker="o", label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(df["Epoch"], df["Wer"], marker="s", linestyle="--", color="green", label="WER")
ax2.plot(df["Epoch"], df["Cer"], marker="s", linestyle="--", color="red", label="CER")
ax2.set_ylabel("WER / CER")
ax2.legend(loc="upper right")

plt.title("Baseline Whisper - Loss & Error Rates")
plt.tight_layout()

plot_path = os.path.join(output_dir, "baseline_whisper_plot.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved plot to: {plot_path}")

print("\nFinal rounded table:")
print(df.to_string(index=False))

Saved table to: SPCS/graph_results/baseline_whisper/baseline_whisper_results.csv
Saved table image to: SPCS/graph_results/baseline_whisper/baseline_whisper_table.png
Saved plot to: SPCS/graph_results/baseline_whisper/baseline_whisper_plot.png

Final rounded table:
 Epoch  Training Loss  Validation Loss    Wer    Cer
     1         2.1561           0.2436 0.2879 0.1534
     2         0.4853           0.0768 0.0910 0.0492
     3         0.1375           0.0443 0.0559 0.0297
     4         0.0149           0.0369 0.0414 0.0189
     5         0.0049           0.0327 0.0314 0.0147
     6         0.0121           0.0302 0.0296 0.0137
     7         0.0084           0.0309 0.0274 0.0122
     8         0.0202           0.0302 0.0278 0.0128
     9         0.0001           0.0307 0.0283 0.0125
    10         0.0001           0.0303 0.0285 0.0130


In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- Results data ----
data = {
    "Epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Training Loss": [4.829699, 1.293808, 0.452286, 0.193986, 0.172611,
                       0.034555, 0.061832, 0.015045, 0.033624, 0.020710],
    "Validation Loss": [0.278970, 0.069809, 0.033190, 0.020631, 0.014249,
                         0.013324, 0.010749, 0.009571, 0.009056, 0.009004],
    "Wer": [0.441176, 0.210846, 0.106985, 0.045404, 0.035662,
            0.030882, 0.026287, 0.026471, 0.020037, 0.022426],
    "Cer": [0.230199, 0.144312, 0.063322, 0.029947, 0.025854,
            0.016933, 0.014449, 0.014729, 0.011650, 0.011895],
}

df = pd.DataFrame(data)

# Round all numeric columns to 3 decimal places
df = df.round(4)

# ---- Output directory ----
output_dir = "SPCS/graph_results/speed_whisper"
os.makedirs(output_dir, exist_ok=True)

# Save table as CSV
csv_path = os.path.join(output_dir, "whisper_speed_aug_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved table to: {csv_path}")

# ---- Save a table image ----
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis("off")
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.title("Whisper Speed Augmentation - Training Results", pad=15)
plt.tight_layout()

table_img_path = os.path.join(output_dir, "whisper_speed_aug_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved table image to: {table_img_path}")

# ---- Save a metrics plot (loss/WER/CER over epochs) ----
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["Epoch"], df["Training Loss"], marker="o", label="Training Loss")
ax1.plot(df["Epoch"], df["Validation Loss"], marker="o", label="Validation Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(df["Epoch"], df["Wer"], marker="s", linestyle="--", color="green", label="WER")
ax2.plot(df["Epoch"], df["Cer"], marker="s", linestyle="--", color="red", label="CER")
ax2.set_ylabel("WER / CER")
ax2.legend(loc="upper right")

plt.title("Whisper Speed Augmentation - Loss & Error Rates")
plt.tight_layout()

plot_path = os.path.join(output_dir, "whisper_speed_aug_plot.png")
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved plot to: {plot_path}")

print("\nFinal rounded table:")
print(df.to_string(index=False))

Saved table to: SPCS/graph_results/speed_whisper/whisper_speed_aug_results.csv
Saved table image to: SPCS/graph_results/speed_whisper/whisper_speed_aug_table.png
Saved plot to: SPCS/graph_results/speed_whisper/whisper_speed_aug_plot.png

Final rounded table:
 Epoch  Training Loss  Validation Loss    Wer    Cer
     1         4.8297           0.2790 0.4412 0.2302
     2         1.2938           0.0698 0.2108 0.1443
     3         0.4523           0.0332 0.1070 0.0633
     4         0.1940           0.0206 0.0454 0.0299
     5         0.1726           0.0142 0.0357 0.0259
     6         0.0346           0.0133 0.0309 0.0169
     7         0.0618           0.0107 0.0263 0.0144
     8         0.0150           0.0096 0.0265 0.0147
     9         0.0336           0.0091 0.0200 0.0116
    10         0.0207           0.0090 0.0224 0.0119
